# Milestone 1: pyKT Data Preparation Pipeline

This notebook prepares the raw English Learning Behaviour Log dataset for Deep Knowledge Tracing using `pykt-toolkit`.

### Objectives:
1. **Filtering & Ordering**: Extract clean answer events chronologically for each learner.
2. **Contiguous ID Mapping**: Map `item_id` and `kc_id` to zero-indexed contiguous integer IDs.
3. **Anti-Leakage Learner Splitting**: Perform a 70/10/20 train/validation/test split by learner ID (840 train / 120 valid / 240 test).
4. **pyKT Bridging**: Format sequences and output padded CSV datasets (`train_valid_sequences.csv`, `test_sequences.csv`) along with `data_config.json`.

In [5]:
# Step 0: Environment Setup & Drive Mounting
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/thesis')
    os.makedirs(PROJECT_ROOT / 'data/raw', exist_ok=True)
    os.chdir(PROJECT_ROOT)
    print(f'Working directory set to: {os.getcwd()}')
except ImportError:
    PROJECT_ROOT = Path('.').resolve()
    print(f'Running locally. Working directory: {PROJECT_ROOT}')

%pip install -q pykt-toolkit pandas numpy scikit-learn

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory set to: /content/drive/MyDrive/thesis


## 1. Prepare Dataset Sequences (`prepare_data.py` Logic)

This step reads `data/raw/english_learning_events.csv`, filters for `answer` events, builds sequence records, and exports ID mappings and split assignments.

In [6]:
import json
import sys
import numpy as np
import pandas as pd

# Configuration & Defaults
INPUT_CSV = PROJECT_ROOT / 'data/raw/english_learning_events.csv'
OUTPUT_DIR = PROJECT_ROOT / 'data/processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 20240116
MAX_SEQ_LEN = 200
SPLIT_RATIOS = {'train': 0.70, 'valid': 0.10, 'test': 0.20}
FOLD_CODE = {'train': 0, 'valid': 1, 'test': 2}

EXPECTED = {'answer_rows': 75006, 'learners': 1200, 'concepts': 96, 'questions': 2189}
EXPECTED_SPLITS = {'train': 840, 'valid': 120, 'test': 240}
REQUIRED_COLUMNS = ['learner_id', 'event_index', 'item_id', 'kc_id', 'kc_dimension', 'event_type', 'timestamp', 'response_time_s', 'correct']

# 1. Load Raw Data
print('Loading raw CSV...')
raw_df = pd.read_csv(INPUT_CSV, encoding='utf-8-sig', dtype={'learner_id': str, 'session_id': str, 'item_id': str, 'kc_id': str, 'kc_dimension': str, 'event_type': str})
raw_df.columns = [c.strip() for c in raw_df.columns]

missing_cols = set(REQUIRED_COLUMNS) - set(raw_df.columns)
if missing_cols:
    raise ValueError(f"Input CSV is missing required columns: {missing_cols}")

# 2. Filter Answer Events
answers_df = raw_df[raw_df['event_type'].str.strip() == 'answer'].copy()
answers_df['correct'] = pd.to_numeric(answers_df['correct'], errors='coerce')
answers_df = answers_df[answers_df['correct'].isin([0, 1])].copy()
answers_df['correct'] = answers_df['correct'].astype(int)
answers_df['event_index'] = answers_df['event_index'].astype(int)
answers_df['timestamp'] = answers_df['timestamp'].astype('int64')
answers_df['response_time_s'] = answers_df['response_time_s'].astype(float)

# Check for duplicate learner-event indices
dup_indices = answers_df.duplicated(subset=['learner_id', 'event_index']).sum()
if dup_indices > 0:
    raise ValueError(f"Found {dup_indices} duplicate (learner_id, event_index) records!")
# Check Item-to-KC mapping consistency (ensure each item maps to exactly 1 KC)
item_kc_counts = answers_df.groupby('item_id')['kc_id'].nunique()
multi_kc_items = item_kc_counts[item_kc_counts > 1]
if len(multi_kc_items) > 0:
    raise ValueError(f"Found items assigned to multiple KCs: {multi_kc_items.to_dict()}")

# Sort chronologically by learner and event_index
answers_df = answers_df.sort_values(['learner_id', 'event_index'], kind='mergesort').reset_index(drop=True)

# 3. Build Contiguous ID Maps
concept_map = answers_df[['kc_id', 'kc_dimension']].drop_duplicates().sort_values('kc_id').reset_index(drop=True)
concept_map.insert(1, 'concepts', np.arange(len(concept_map), dtype=int))

question_map = answers_df[['item_id', 'kc_id']].drop_duplicates().sort_values('item_id').reset_index(drop=True)
question_map.insert(1, 'questions', np.arange(len(question_map), dtype=int))
question_map = question_map.merge(concept_map, on='kc_id', how='left')[['item_id', 'questions', 'kc_id', 'concepts', 'kc_dimension']]

# 4. Build Sequences per Learner
q_idx = dict(zip(question_map['item_id'], question_map['questions']))
c_idx = dict(zip(concept_map['kc_id'], concept_map['concepts']))
answers_df['questions'] = answers_df['item_id'].map(q_idx)
answers_df['concepts'] = answers_df['kc_id'].map(c_idx)

rows = []
learner_ids = sorted(answers_df['learner_id'].unique())
uid_of = {lid: i for i, lid in enumerate(learner_ids)}
join_str = lambda vals: ','.join(str(v) for v in vals)

for lid, g in answers_df.groupby('learner_id', sort=True):
    if len(g) > MAX_SEQ_LEN:
        g = g.iloc[:MAX_SEQ_LEN]
    n = len(g)
    rows.append({
        'uid': uid_of[lid],
        'questions': join_str(g['questions'].astype(int)),
        'concepts': join_str(g['concepts'].astype(int)),
        'responses': join_str(g['correct'].astype(int)),
        'timestamps': join_str(g['timestamp'].astype('int64')),
        'usetimes': join_str(round(float(x), 2) for x in g['response_time_s']),
        'selectmasks': join_str(['1'] * n),
        'learner_id': lid,
        'seq_len': n
    })
seq_df = pd.DataFrame(rows)

# 5. Assign Learner-Level Splits (70/10/20 anti-leakage)
n = len(seq_df)
n_train = int(round(SPLIT_RATIOS['train'] * n))
n_valid = int(round(SPLIT_RATIOS['valid'] * n))
rng = np.random.RandomState(SEED)
perm = rng.permutation(n)
labels = np.empty(n, dtype=object)
labels[perm[:n_train]] = 'train'
labels[perm[n_train:n_train + n_valid]] = 'valid'
labels[perm[n_train + n_valid:]] = 'test'
seq_df['split'] = labels
seq_df['fold'] = seq_df['split'].map(FOLD_CODE)

# Verify counts against expected metadata
actual_counts = {
    'answer_rows': len(answers_df),
    'learners': seq_df['learner_id'].nunique(),
    'concepts': len(concept_map),
    'questions': len(question_map)
}
for k, exp in EXPECTED.items():
    if actual_counts[k] != exp:
        raise ValueError(f"Count mismatch for {k}: expected {exp}, got {actual_counts[k]}")
split_counts = {s: sum(seq_df['split'] == s) for s in EXPECTED_SPLITS}
if split_counts != EXPECTED_SPLITS:
    raise ValueError(f"Split mismatch: expected {EXPECTED_SPLITS}, got {split_counts}")

# Write Outputs
out_cols = ['fold', 'uid', 'questions', 'concepts', 'responses', 'timestamps', 'usetimes', 'selectmasks', 'learner_id', 'split', 'seq_len']
seq_df[out_cols].sort_values('uid').to_csv(OUTPUT_DIR / 'pykt_formatted_sequences.csv', index=False)
question_map.to_csv(OUTPUT_DIR / 'question_map.csv', index=False)
concept_map.to_csv(OUTPUT_DIR / 'concept_map.csv', index=False)
seq_df[['learner_id', 'uid', 'split', 'fold', 'seq_len']].sort_values('uid').to_csv(OUTPUT_DIR / 'split_assignments.csv', index=False)

print('=' * 60)
print('DATA PREPARATION COMPLETE & VERIFIED')
print(f'Retained Answer Rows: {len(answers_df):,}')
print(f'Unique Learners     : {seq_df["learner_id"].nunique():,}')
print(f'Unique Concepts (KC): {len(concept_map):,}')
print(f'Unique Questions    : {len(question_map):,}')
# print(f'Learner Splits      : Train={sum(seq_df["split"]=="train")}, Valid={sum(seq_df["split"]=="valid")}, Test={sum(seq_df["split"]=="test")}')
print(f'Learner Splits      : Train={split_counts["train"]}, Valid={split_counts["valid"]}, Test={split_counts["test"]}')
print('=' * 60)

Loading raw CSV...
DATA PREPARATION COMPLETE & VERIFIED
Retained Answer Rows: 75,006
Unique Learners     : 1,200
Unique Concepts (KC): 96
Unique Questions    : 2,189
Learner Splits      : Train=840, Valid=120, Test=240


## 2. Initialize pyKT Dataset & Data Config (`init_pykt_dataset.py` Logic)

This step formats sequences for pyKT's `KTDataset` class by padding with `-1` to `maxlen=200`, converting `usetimes` to integer milliseconds, and generating `data_config.json`.

In [7]:
import json
import sys
import pandas as pd

PYKT_DATA_DIR = PROJECT_ROOT / 'data/processed/pykt_english'
CONFIG_DIR = PROJECT_ROOT / 'knowledge_tracing/configs'
PYKT_DATA_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
SEQ_FIELDS = ['questions', 'concepts', 'responses', 'timestamps', 'usetimes', 'selectmasks']
M1_FOLDS = {0: 'train', 1: 'valid', 2: 'test'}
EXPECTED_LEARNERS = {'train': 840, 'valid': 120, 'test': 240}
PAD = -1
MAXLEN = 200

formatted_csv = OUTPUT_DIR / 'pykt_formatted_sequences.csv'
if not formatted_csv.exists():
    raise FileNotFoundError(f'{formatted_csv} not found; run section 1 first')
df = pd.read_csv(formatted_csv)
required_columns = ['fold', 'uid', 'learner_id', *SEQ_FIELDS]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f'Input is missing columns: {missing_columns}')
if not set(df['fold'].unique()) <= set(M1_FOLDS):
    raise ValueError(f'Unexpected fold values: {sorted(df["fold"].unique())}')
if df['learner_id'].duplicated().any():
    raise ValueError('A learner appears in more than one row; splits could leak')
split_sizes = {name: int((df['fold'] == fold).sum()) for fold, name in M1_FOLDS.items()}
print(f'Learners per fold: {split_sizes}')
if len(df) == 1200 and split_sizes != EXPECTED_LEARNERS:
    raise ValueError(f'Split sizes differ from expected {EXPECTED_LEARNERS}: {split_sizes}')

def to_int_list(row, field):
    parts = str(row[field]).split(',')
    if field == 'usetimes':
        return [int(round(float(value) * 1000)) for value in parts]
    return [int(value) for value in parts]

def pad_seq(values, maxlen):
    if len(values) > maxlen:
        raise ValueError(f'Sequence length {len(values)} exceeds maxlen={maxlen}')
    return ','.join(str(value) for value in values + [PAD] * (maxlen - len(values)))

out_rows = []
for _, row in df.iterrows():
    sequences = {field: to_int_list(row, field) for field in SEQ_FIELDS}
    seq_len = len(sequences['responses'])
    if any(len(values) != seq_len for values in sequences.values()):
        raise ValueError(f'Learner {row["learner_id"]} has sequence fields of different lengths')
    if set(sequences['responses']) - {0, 1}:
        raise ValueError(f'Learner {row["learner_id"]} has non-binary responses')
    fold = {0: 0, 1: 1, 2: -1}[int(row['fold'])]
    out_rows.append({
        'fold': fold,
        'uid': int(row['uid']),
        **{field: pad_seq(sequences[field], MAXLEN) for field in SEQ_FIELDS},
        'learner_id': row['learner_id']
    })
out_df = pd.DataFrame(out_rows)
cols = ['fold', 'uid', *SEQ_FIELDS, 'learner_id']

for stale_cache in PYKT_DATA_DIR.glob('*.pkl'):
    stale_cache.unlink()
out_df[out_df['fold'].isin([0, 1])][cols].to_csv(PYKT_DATA_DIR / 'train_valid_sequences.csv', index=False)
out_df[out_df['fold'] == -1][cols].to_csv(PYKT_DATA_DIR / 'test_sequences.csv', index=False)

question_map_path = OUTPUT_DIR / 'question_map.csv'
concept_map_path = OUTPUT_DIR / 'concept_map.csv'
if not question_map_path.exists() or not concept_map_path.exists():
    raise FileNotFoundError('question_map.csv and concept_map.csv must exist from section 1')
num_q = len(pd.read_csv(question_map_path))
num_c = len(pd.read_csv(concept_map_path))
max_q = max(int(value) for seq in df['questions'] for value in str(seq).split(','))
max_c = max(int(value) for seq in df['concepts'] for value in str(seq).split(','))
if max_q >= num_q or max_c >= num_c:
    raise ValueError('Sequence IDs exceed the question/concept maps')

data_config = {
    'english_kt': {
        'dpath': 'data/processed/pykt_english',
        'num_q': num_q,
        'num_c': num_c,
        'max_concepts': 1,
        'input_type': ['questions', 'concepts'],
        'maxlen': MAXLEN,
        'emb_path': '',
        'train_valid_file': 'train_valid_sequences.csv',
        'test_file': 'test_sequences.csv',
        'folds': [0, 1],
        'fold_meaning': {'0': 'train', '1': 'valid', '-1': 'test (separate file)'}
    }
}
(CONFIG_DIR / 'data_config.json').write_text(json.dumps(data_config, indent=2))
bridge_manifest = {
    'source': str(formatted_csv),
    'learners_per_split': split_sizes,
    'maxlen': MAXLEN,
    'pad_value': PAD,
    'usetimes_unit': 'milliseconds (integer)',
    'num_q': num_q,
    'num_c': num_c
}
(PYKT_DATA_DIR / 'bridge_manifest.json').write_text(json.dumps(bridge_manifest, indent=2))

print(f'num_q={num_q}  num_c={num_c}  maxlen={MAXLEN}')
print(f'Wrote {PYKT_DATA_DIR / "train_valid_sequences.csv"} ({split_sizes["train"] + split_sizes["valid"]} train/validation learners)')
print(f'Wrote {PYKT_DATA_DIR / "test_sequences.csv"} ({split_sizes["test"]} test learners)')
print(f'Wrote {CONFIG_DIR / "data_config.json"}')
print(f'Wrote {PYKT_DATA_DIR / "bridge_manifest.json"}')

Learners per fold: {'train': 840, 'valid': 120, 'test': 240}
num_q=2189  num_c=96  maxlen=200
Wrote /content/drive/MyDrive/thesis/data/processed/pykt_english/train_valid_sequences.csv (960 train/validation learners)
Wrote /content/drive/MyDrive/thesis/data/processed/pykt_english/test_sequences.csv (240 test learners)
Wrote /content/drive/MyDrive/thesis/knowledge_tracing/configs/data_config.json
Wrote /content/drive/MyDrive/thesis/data/processed/pykt_english/bridge_manifest.json
